# Loop0, stage 03: background cutoffs and logicle parameters

For each antibody this measures where real staining begins, using the control
that leaves that antibody out, and turns that into the parameters of the logicle
transformation.

The control cells alone are not enough.  The logicle transformation also needs
the top of the display range, which comes from the stained cells of the screen.
Both are concatenated here, the controls labelled by the antibody they leave out
and the screen labelled `fully_stained`.

## Inputs

| file | produced by | holds |
|---|---|---|
| `Loop0_FMOs_Gated.h5ad` | stage 02b | the gated control cells |
| `Loop0_Gated_ExpRem.h5ad` | stage 02 | the gated cells of the screen |

## Outputs

| file | holds |
|---|---|
| `marker_cutoff.pkl` | one background cutoff per antibody |
| `logicle_parameters.json` | the four logicle parameter lists `t`, `m`, `w` and `a` |

Both outputs are used by stage 04 of every loop, not only this one.  The other
loops carry a copy of these two files rather than recomputing them.

Run the cells from top to bottom.

In [ ]:
import json
import math
import pickle

import anndata as ann
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sb
from scipy.stats import gaussian_kde

IN_FMO_GATED = "Loop0_FMOs_Gated.h5ad"
IN_SCREEN = "Loop0_Gated_ExpRem.h5ad"

OUT_CUTOFFS = "marker_cutoff.pkl"
OUT_LOGICLE = "logicle_parameters.json"

# Everything above this percentile of a control channel is treated as outlier
# noise rather than background, and dropped before the maximum is taken.
FMO_PERCENTILE = 99.5
# The same idea for the stained cells, which set the top of the display range.
STAINED_PERCENTILE = 99.75

In [ ]:
adata = sc.read(IN_FMO_GATED)
antibody_list = adata.var["antibody"].unique().tolist()
print(f"{len(antibody_list)} antibodies:", antibody_list)

## Add the stained cells of the screen

Their `fmo` label is `fully_stained`, which is how the loop below tells them
apart from the controls.

In [ ]:
screen = sc.read(IN_SCREEN)
screen.obs["fmo"] = "fully_stained"

adata = ann.concat(
    [adata, screen], join="inner", merge="same", uns_merge="unique",
    label="batch", index_unique="-",
)
del screen
adata.obs["fmo"].value_counts()

## Measure the background

Each panel shows one antibody.  The red histogram is the control that leaves that
antibody out, so it is background.  The black histogram is the stained cells.
The title gives the highest background intensity, which becomes that antibody's
cutoff.

In [ ]:
def dense(view):
    """Return the single-channel intensities of `view` as a flat float array."""
    X = view.X
    return np.asarray(X.toarray() if hasattr(X, "toarray") else X).ravel()


marker_cutoff = {}
t, m, w, a = [], [], [], []

stained = adata[(adata.obs["fmo"] == "fully_stained").values, :].copy()
fig, axes = plt.subplots(len(antibody_list), figsize=(6, 3 * len(antibody_list)))

for i, marker in enumerate(antibody_list):
    channel = (adata.var["antibody"] == marker).values

    # The control that leaves this antibody out.  Its signal in this channel is
    # background, so its maximum is where real staining begins.
    control = dense(adata[(adata.obs["fmo"] == marker).values, :][:, channel])
    control = control[control <= np.percentile(control, FMO_PERCENTILE)]
    top_of_background = float(control.max())

    # The stained cells set the top of the display range.
    values = dense(stained[:, channel])
    values = values[values <= np.percentile(values, STAINED_PERCENTILE)]
    top_of_scale = float(values.max())

    if top_of_background <= 0 or top_of_scale <= 0:
        raise ValueError(f"{marker}: a maximum is not positive, so it has no logarithm")

    marker_cutoff[marker] = top_of_background
    t.append(top_of_scale)                     # top of the display range
    m.append(math.log10(top_of_scale))         # decades the display spans
    w.append(math.log10(top_of_background))    # decades compressed near zero
    a.append(-math.log10(top_of_background))   # extra negative decades

    xmin, xmax = -3 * control.max(), 6 * control.max()
    bins = np.linspace(xmin, xmax, 41)
    sb.histplot(control, bins=bins, ax=axes[i], stat="density", color="red")
    sb.histplot(values, bins=bins, ax=axes[i], stat="density", color="black")
    x_eval = np.linspace(xmin, xmax, 500)
    axes[i].plot(x_eval, gaussian_kde(control)(x_eval), color="red")
    axes[i].set_title(f"FMO = {marker}, background reaches {control.max():.1f}")
    axes[i].set_xlim(xmin, xmax)
    axes[i].set_xlabel("intensity")

plt.tight_layout()
plt.show()
marker_cutoff

## Write the two calibration files

In [ ]:
with open(OUT_CUTOFFS, "wb") as handle:
    pickle.dump(marker_cutoff, handle)

with open(OUT_LOGICLE, "w") as handle:
    json.dump({"t": t, "m": m, "w": w, "a": a}, handle, indent=2)
print(f"wrote {OUT_CUTOFFS} with {len(marker_cutoff)} antibodies")
print(f"wrote {OUT_LOGICLE} with {len(t)} channels")